# Support Vector Machines (SVM) — From Intuition to Code

We've covered Linear Regression, Logistic Regression, Naive Bayes, Decision Trees, and Random Forests. 

Today, we meet one of the most elegant and mathematically beautiful algorithms in all of Machine Learning: **The Support Vector Machine (SVM)**.

If Logistic Regression is about drawing *any* line that separates two classes, an SVM is about drawing the **BEST possible line**, with the widest possible "street" between the classes.

Let's dive in.

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("Libraries imported!")

Libraries imported!


---
## 1. The Intuition: The Widest Street

Imagine you have a table where red apples are on the left and green apples are on the right. You want to place a stick on the table to separate them.

- **Logistic Regression** will place the stick anywhere as long as it separates the reds from the greens.
- **SVM** looks at the apples closest to the middle, and places the stick exactly halfway between the innermost red apple and the innermost green apple. It tries to build the **widest possible "street"** between the two classes.

### Key Terminology:
1. **Hyperplane**: The decision boundary (a line in 2D, a flat sheet in 3D, etc.).
2. **Margin**: The width of the "street" separating the classes.
3. **Support Vectors**: The data points that sit exactly on the edge of the street. They are called "support" vectors because they alone dictate where the line goes. If you delete all the other points far away from the street, the line wouldn't change at all!

---
## 2. Hard Margin vs. Soft Margin (The `C` Hyperparameter)

What happens if there's a single green apple sitting way over on the red side? 

- **Hard Margin SVM**: Refuses to let any apple cross the street. It will twist and turn the boundary aggressively just to classify every single training point perfectly. (Extremely sensitive to outliers -> **Overfitting**).
- **Soft Margin SVM**: Says, *"It's okay if a few apples are on the wrong side, as long as the overall street is nice and wide."*

In Scikit-Learn, this is controlled by the hyperparameter **`C`**:
- **High `C`** (e.g., C=1000): Strict! Narrow margin, very few mistakes allowed (High risk of overfitting).
- **Low `C`** (e.g., C=0.1): Relaxed! Wider margin, allows more mistakes on the training data (Better generalization).

---
## 3. The Math and The Loss Function: Hinge Loss

For Logistic Regression, we used Binary Cross-Entropy (Log Loss). 
For SVM, we use **Hinge Loss**.

In SVMs for binary classification, our labels are usually represented as **$y_{i} \in \{-1, 1\}$** instead of 0 and 1.

The SVM makes predictions based on the sign of $w \cdot x + b$:
- If $w \cdot x + b > 0 \implies \text{predict } +1$
- If $w \cdot x + b < 0 \implies \text{predict } -1$

### The Hinge Loss equation:
$$\text{Loss}_i = \max(0, 1 - y_i (w \cdot x_i + b))$$

**How it works:**
1. **$y_{i} (w \cdot x_i + b) \ge 1$**: The point is correctly classified AND is safely outside the street. Loss = 0! The model literally stops caring about this point. This is why only the Support Vectors matter.
2. **$0 < y_{i} (w \cdot x_i + b) < 1$**: The point is correctly classified, but it's inside the street (in the margin). Small penalty.
3. **$y_{i} (w \cdot x_i + b) \le 0$**: The point is incorrectly classified (wrong side of the boundary). Huge penalty.

### The Full Objective Function:
To train an SVM, we want to minimize the Hinge Loss **PLUS** a regularization term that forces the street to be as wide as possible.

$$ J(w, b) = \frac{1}{2} ||w||^2 + C \sum_{i=1}^n \max(0, 1 - y_i(w \cdot x_i + b)) $$

- **Minimize $\frac{1}{2} \|w\|^2$**: This makes the margin wider.
- **Minimize Hinge Loss**: This reduces classification errors.
- **$C$**: Controls the trade-off. 

---
## 4. Building a Linear SVM from Scratch (with Gradient Descent)

Let's implement this objective function using Gradient Descent. We will compute the gradients of the Hinge Loss.

In [ ]:
class MeraLinearSVM:
    def __init__(self, learning_rate=0.001, lambda_param=0.01, n_iters=1000):
        self.lr = learning_rate
        self.lambda_param = lambda_param  # This is inversely proportional to C (lambda = 1/C)
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        # Enforce y to be -1 or 1
        y_ = np.where(y <= 0, -1, 1)
        
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        # Gradient Descent
        for _ in range(self.n_iters):
            for i, x_i in enumerate(X):
                # Condition: y_i * (w.x + b) >= 1 (Point is correct and outside the margin)
                condition = y_[i] * (np.dot(x_i, self.w) - self.b) >= 1
                if condition:
                    # Only the regularization term contributes to the gradient
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Both regularization and hinge loss contribute to the gradient
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_[i]))
                    self.b -= self.lr * y_[i]

    def predict(self, X):
        approx = np.dot(X, self.w) - self.b
        # Convert back to 0 and 1
        return np.where(np.sign(approx) == -1, 0, 1)

print("MeraLinearSVM defined!")

---
## 5. Testing the Implementation and Visualizing the Margin

In [ ]:
from sklearn.datasets import make_blobs

# Create a perfectly linearly separable dataset
X, y = make_blobs(n_samples=250, centers=2, random_state=42, cluster_std=1.2)

# Train our custom SVM
mera_svm = MeraLinearSVM(learning_rate=0.001, lambda_param=0.01, n_iters=1000)
mera_svm.fit(X, y)

def plot_svm_boundary(model, X, y):
    # Create a meshgrid
    x0_min, x0_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x1_min, x1_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx0, xx1 = np.meshgrid(np.linspace(x0_min, x0_max, 200),
                           np.linspace(x1_min, x1_max, 200))
    
    # Calculate the decision function w*x - b
    Z = np.dot(np.c_[xx0.ravel(), xx1.ravel()], model.w) - model.b
    Z = Z.reshape(xx0.shape)
    
    plt.figure(figsize=(10, 6))
    plt.contourf(xx0, xx1, np.where(Z<0, 0, 1), alpha=0.3, cmap='coolwarm')
    plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k')
    
    # Plot the decision boundary and margins
    # Decision boundary: w*x - b = 0
    plt.contour(xx0, xx1, Z, colors='k', levels=[-1, 0, 1], alpha=0.8, 
                linestyles=['--', '-', '--'])
    
    plt.title("MeraLinearSVM - Decision Boundary and Margins")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.show()

plot_svm_boundary(mera_svm, X, y)

Look closely at the plot:
- The **solid black line** is the decision boundary ($w \cdot x + b = 0$).
- The **dashed lines** are the margins ($w \cdot x + b = 1$ and $w \cdot x + b = -1$).
- The data points resting directly on or inside the dashed lines are the **Support Vectors**.

---
## 6. The Magic of SVMs: The Kernel Trick

A Linear SVM is great, but what if the data isn't linearly separable? (Like our inner/outer circle dataset from the Random Forest notebook).

If you try to draw a straight line through concentric circles, you will fail.

But SVMs have a superpower called **The Kernel Trick**.
If the data can't be separated by a line in 2D, SVMs mathematically project the data into a **higher dimension** (like 3D), where it suddenly *can* be sliced by a flat sheet (a hyperplane). Then, it projects that slice back down to 2D, where it looks like a complex curve!

The math is so clever that the SVM never actually has to compute the expensive 3D coordinates. It just uses a "Kernel function" to calculate relationships in the higher dimension.

### Common Kernels:
1. **Linear**: Basic straight line.
2. **Polynomial**: Projects into polynomial curves.
3. **RBF (Radial Basis Function / Gaussian)**: The default. Acts like a topographic map, drawing complex, enclosed boundaries around clusters of data.

In [ ]:
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Generate the non-linear circles dataset
X_circle, y_circle = make_circles(n_samples=400, noise=0.1, factor=0.3, random_state=42)

# 1. Linear Kernel (Will fail)
svm_linear = SVC(kernel='linear')
svm_linear.fit(X_circle, y_circle)

# 2. RBF Kernel (Will succeed brilliantly)
svm_rbf = SVC(kernel='rbf', C=1.0, gamma='scale')
svm_rbf.fit(X_circle, y_circle)

def plot_kernel_comparison(model1, model2, X, y, title1, title2):
    x0_min, x0_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    x1_min, x1_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx0, xx1 = np.meshgrid(np.linspace(x0_min, x0_max, 200),
                           np.linspace(x1_min, x1_max, 200))
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    for i, (model, title) in enumerate(zip([model1, model2], [title1, title2])):
        preds = model.predict(np.c_[xx0.ravel(), xx1.ravel()]).reshape(xx0.shape)
        
        axes[i].contourf(xx0, xx1, preds, alpha=0.3, cmap='coolwarm')
        axes[i].scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolors='k')
        axes[i].set_title(f"{title} (Acc: {accuracy_score(y, model.predict(X)):.2%})")
        
    plt.show()

plot_kernel_comparison(svm_linear, svm_rbf, X_circle, y_circle, 
                       "Linear Kernel (Fails on Non-Linear Data)", 
                       "RBF Kernel (Succeeds via Kernel Trick)")

Notice how the **RBF Kernel** perfectly wraps around the inner class! It projected the data into infinite dimensions (yes, literally infinite), sliced it, and projected it back as a circle.

*Note on Hyperparameters for RBF:*
- `C`: Still the soft-margin parameter. Higher C = stricter, more complex boundary.
- `gamma`: Defines how far the influence of a single training example reaches. Low `gamma` = far reach (smooth boundary). High `gamma` = close reach (very tightly wraps around individual points -> overfitting).

---
## 7. SVMs vs. Logistic Regression

Both are fundamentally linear classifiers (before the Kernel Trick). When to use which?

1. **Logistic Regression** outputs probabilities natively. (e.g., "I am 82% sure this is Spam"). SVMs output distances to a margin, which don't directly map to probabilities.
2. **SVMs** only care about the Support Vectors. Outliers far away from the margin don't affect an SVM at all. Logistic Regression cares about *every* point, so an intense outlier can skew the Logistic boundary.
3. **The Kernel Trick**: This is the major advantage of SVMs. If you know your data has complex, non-linear boundaries and you don't have enough data for a deep neural network, an SVM with an RBF kernel is a powerhouse.

---
## 8. When and Why to Use SVMs

### Best used for:
- **High-Dimensional Data**: Even when the number of features exceeds the number of samples (like genomics or text classification), SVMs perform exceptionally well.
- **Complex Non-Linear Boundaries**: Thanks to the Kernel Trick.
- **When Outliers are Present**: SVMs ignore data far from the margin.
  
### Avoid when:
- **Massive Datasets** ($>100,000$ rows): Training an SVM with a non-linear kernel is an $O(n^2)$ or $O(n^3)$ math problem. It gets **extremely slow** on huge datasets. (Random Forests or Neural Networks are better here).
- **Extremely Noisy Data**: If the classes heavily overlap with no clear separation, SVMs struggle to find a good margin, leading to poor generalization.

### Benefits:
1. **Versatile**: Different kernels provide massive flexibility.
2. **Memory Efficient**: Once trained, it only needs to "remember" the support vectors to make predictions, not the whole dataset.
3. **Mathematical Guarantees**: Unlike Neural nets which get trapped in local minimums, the SVM optimization problem is convex — it's guaranteed to find the absolute global best margin.

### Bottlenecks:
1. **Slow Training Phase**: RBF kernel training grinds to a halt on very large datasets.
2. **Feature Scaling is Mandatory**: Because SVM works by calculating distances (margins) between points, features with large scales will dominate. **You MUST standard scale your data before feeding it to an SVM.**
3. **Hyperparameter Sensitivity**: Getting the exact right combination of `C` and `gamma` often requires careful tuning.